# 2. Experiments with models

## Model A: Logistic Regression - base model

### *OPTIONAL* **(1) Validation**:

One stratified train/validation split.

See (2) Cross-Validation below.

The training is performed in the cell below:

In [ ]:

import pandas as pd
import numpy as np

In [ ]:
clean_df = pd.read_csv('final_clean_train_df.csv')

In [ ]:
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score


# Features X, and target labels y
X = clean_df.drop(columns=["readmitted"])
y = clean_df["readmitted"]   # values are: "NO", ">30", "<30"

# Split into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    stratify=y,         # to maintain class proportions (important for <30)
    random_state=42     # makes the split reproducible
)

# Build pipeline model
model = Pipeline(steps=[           # pipeline chains preprocessing + model into one object.
    ("scaler", StandardScaler()),  # step 1: standardises each feature to mean 0, std 1.
    ("clf", LogisticRegression(    # step 2: train logistic regression (the classifier) on the scaled data
            # multi_class="multinomial",   # uses true softmax/multinomial loss for 3 classes
            solver="lbfgs",              # strong default optimiser for multinomial + L2 regularisation
            max_iter=500,                # increase max iters in case of non-convergence 
            class_weight="balanced",     # reweights classes inversely proportional to their frequency, so the minority class matters more
            n_jobs=None                  # lbfgs ignores n_jobs; OK to leave
            )
    )
])

model.fit(X_train, y_train) # Fits logistic regression weights on transformed X_train.

# Evaluate model
y_pred = model.predict(X_val)

print("\nBalanced accuracy:", balanced_accuracy_score(y_val, y_pred)) # The best value is 1 and the worst value is 0 when adjusted=False. It is defined as the mean of recalls (=sensitivity) (obtained on each class) across the three classess. Avoids inflated performance estimates on imbalanced datasets.
print("Macro F1:", f1_score(y_val, y_pred, average="macro")) # Mean of per-class F1, weighting each class equally. F1 score reaches its best value at 1 and worst score at 0. Can be interpreted as a harmonic mean of the precision (=accuracy of positive predictions) and recall (=sensitivity). 

print("\nConfusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_val, y_pred, labels=["NO", ">30", "<30"])) # Rows are true labels, Columns are predicted labels:
                                                                    #   True NO predicted as:  NO / >30 / <30
                                                                    #   True >30 predicted as: NO / >30 / <30
                                                                    #   True <30 predicted as: NO / >30 / <30

print("\nClassification report:")
print(classification_report(y_val, y_pred, digits=4)) # Per-class precision/recall/F1, plus macro/weighted averages.



Balanced accuracy: 0.4562591297104652
Macro F1: 0.4326890589423966

Confusion matrix (rows=true, cols=pred):
[[5650 2325 1896]
 [2075 2538 1785]
 [ 631  599  819]]

Classification report:
              precision    recall  f1-score   support

         <30     0.1820    0.3997    0.2501      2049
         >30     0.4647    0.3967    0.4280      6398
          NO     0.6762    0.5724    0.6200      9871

    accuracy                         0.4917     18318
   macro avg     0.4409    0.4563    0.4327     18318
weighted avg     0.5470    0.4917    0.5115     18318



#### Output 
**Balanced accuracy:** 0.4562591297104652

**Macro F1:** 0.4326890589423966


**Confusion matrix (rows=true, cols=pred):**
| True \ Pred | NO   | >30  | <30  |
|-------------|------|------|------|
| True NO     | 5650 | 2325 | 1896 |
| True >30    | 2075 | 2538 | 1785 |
| True <30    | 631  | 599  | 819  |

**Classification report:**
| Class | Precision | Recall | F1-score | Support |
|-------|-----------|--------|----------|---------|
| <30   | 0.1820    | 0.3997 | 0.2501   | 2049    |
| >30   | 0.4647    | 0.3967 | 0.4280   | 6398    |
| NO    | 0.6762    | 0.5724 | 0.6200   | 9871    |
|       |           |        |          |         |
| Accuracy |        |        | 0.4917   | 18318   |
| Macro Avg | 0.4409 | 0.4563 | 0.4327 | 18318   |
| Weighted Avg | 0.5470 | 0.4917 | 0.5115 | 18318 |


________________________________________

#### Explanation:

##### 1) Balanced accuracy: 0.4563
Balanced accuracy is:
(Recall_NO + Recall_(>30) + Recall_< 30) / 3

From your classification report:
- Recall <30 = 0.3997
- Recall >30 = 0.3967
- Recall NO = 0.5724

Average: 

(0.3997 + 0.3967 + 0.5724) / 3 ≈ 0.4563

**Interpretation:** the model’s average ability to correctly catch each class is about 45.6%, which is better than random (random would be ~33% balanced accuracy) but not strong.
 
##### 2) Macro F1: 0.4327

Macro F1 averages F1 across classes:
- <30 F1 = 0.2501
- '>30 F1 = 0.4280
- NO F1 = 0.6200

Macro average:
(0.2501 + 0.4280 + 0.6200) / 3 ≈ 0.4327

**Interpretation:** performance is dragged down heavily by the <30 class, which has low precision.
 
#####  3) Confusion matrix

You printed with labels ["NO", ">30", "<30"], so rows are true class, columns are predicted class:
- [[5650 2325 1896]   True NO predicted as:  NO / >30 / <30
- [2075 2538 1785]   True >30 predicted as: NO / >30 / <30
- [ 631  599  819]]  True <30 predicted as: NO / >30 / <30
 
Let’s interpret each row:

- True NO (9871 total):
    - Correct NO: 5650
    - Misclassified as >30: 2325
    - Misclassified as <30: 1896
- Recall(NO) = 5650/9871 = 0.5724 (matches report)

- True >30 (6398 total)
    - Correct >30: 2538
    - Misclassified as NO: 2075
    - Misclassified as <30: 1785
- Recall(>30) = 2538/6398 = 0.3967

- True <30 (2049 total)
    - Correct <30: 819
    - Misclassified as NO: 631
    - Misclassified as >30: 599
- Recall(<30) = 819/2049 = 0.3997

**Big picture:** the model is confused between the two readmission classes and NO. Many NO cases are predicted as readmission, and many readmissions predicted as NO.

#####  4) Classification report (precision vs recall intuition)
<30
- Precision 0.1820 (very low): when the model predicts <30, it’s right only ~18% of the time.
- Recall 0.3997: it finds ~40% of true <30 cases.

This happens often when class_weight="balanced" pushes the model to “try harder” on the minority class, increasing recall at the expense of precision.

NO
- Precision 0.6762: predicted NOs are fairly reliable.
- Recall 0.5724: but it misses many true NOs (labels them as readmitted).

**Overall accuracy 0.4917**

Accuracy is ~49%, but because of imbalance, it’s less informative than macro F1 / balanced accuracy.
 



### **(2) Cross-Validation: V1** - With StandardScalar preprocessing only
-> Uses StandardScaler to normalise all features to mean 0, std 1.

In [ ]:
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score, balanced_accuracy_score

# Load cleaned data
DATA_PATH = Path("..") / "Dataset" / "clean_train_df.csv"
clean_df = pd.read_csv(DATA_PATH)

# Features X, and target labels y
X = clean_df.drop(columns=["readmitted"])
y = clean_df["readmitted"]   # values are: "NO", ">30", "<30"

# Build pipeline model
model = Pipeline(steps=[           # pipeline chains preprocessing + model into one object.
    ("scaler", StandardScaler()),  # step 1: standardises each feature to mean 0, std 1.
    ("clf", LogisticRegression(    # step 2: train logistic regression (the classifier) on the scaled data
            solver="lbfgs",              # optimisation method (default) to find best coefficients for multinomial + L2 regularisation
            max_iter=500,                # enough iterations for convergence in a high-dimensional dataset
            class_weight="balanced",     # increases the penalty for misclassifying minority classes (reweights classes inversely proportional to their frequency)
            n_jobs=None                  # lbfgs ignores n_jobs; OK to leave
            )
    )
])

# Cross-validation
cv = StratifiedKFold(
    n_splits=10, 
    shuffle=True, 
    random_state=42
)

# Define evaluation metrics - what to compute on each fold's validation set
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": make_scorer(balanced_accuracy_score),
    "f1_macro": make_scorer(f1_score, average="macro"),
    "f1_weighted": make_scorer(f1_score, average="weighted")
}

# Run cross-validation (= training + validation). There are 10 separate training runs.
cv_results = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)


# Summary results
print("=== 10-Fold CV Logistic Regression Summary ===")
for metric in scoring.keys():
    scores = cv_results[f"test_{metric}"]
    print(
        f"{metric}: Mean={scores.mean():.4f}, Std={scores.std():.4f}"
    )

=== 10-Fold CV Logistic Regression Summary ===
accuracy: Mean=0.4951, Std=0.0048
balanced_accuracy: Mean=0.4576, Std=0.0053
f1_macro: Mean=0.4348, Std=0.0043
f1_weighted: Mean=0.5145, Std=0.0047


In [32]:
# Scoring metric: [Results for 10 folds]
cv_results


{'fit_time': array([1.50130606, 1.10355973, 1.39485431, 1.3937881 , 1.42553806,
        1.85051608, 3.58783102, 1.39068389, 1.15696001, 1.58023405]),
 'score_time': array([0.09454799, 0.07329893, 0.06069589, 0.07873702, 0.08403707,
        0.12461376, 0.07118487, 0.07458401, 0.06933188, 0.07095003]),
 'test_accuracy': array([0.4956873 , 0.50049132, 0.49514139, 0.49132001, 0.48629763,
        0.50267496, 0.49470466, 0.48902719, 0.49699749, 0.49781612]),
 'train_accuracy': array([0.50760645, 0.50772777, 0.50625986, 0.50850419, 0.50868616,
        0.50783695, 0.50635691, 0.50914716, 0.5079704 , 0.50851015]),
 'test_balanced_accuracy': array([0.45746605, 0.46264014, 0.4618102 , 0.46007259, 0.44404978,
        0.46179913, 0.45996996, 0.4527383 , 0.45626219, 0.45796491]),
 'train_balanced_accuracy': array([0.47807011, 0.47713275, 0.47724476, 0.47777662, 0.47814875,
        0.47811943, 0.47611474, 0.47958513, 0.47830725, 0.4780675 ]),
 'test_f1_macro': array([0.43435621, 0.4395484 , 0.4359656

#### Output 

**Reported results**

| Metric              | Mean   | Std    |
|---------------------|--------|--------|
| Accuracy            | 0.4951 | 0.0048 |
| Balanced Accuracy   | 0.4576 | 0.0053 |
| F1 (Macro)          | 0.4348 | 0.0043 |
| F1 (Weighted)       | 0.5145 | 0.0047 |



________________________________________

#### Explanation:

##### Interpretation
- Low standard deviation → stable but limited performance

- Model performs better than random (≈0.33 for 3 classes).

- Balanced accuracy and macro F1 are lower → minority classes are harder to predict.

- This is expected for this dataset (known to be noisy and imbalanced).

**Conclusion:**
✔️ Your results are reasonable for a baseline logistic regression.


##### *Details:
**1) Accuracy ≈ 49.5%**

- About half of patients are classified correctly.

- For a 3-class, imbalanced, noisy clinical dataset, this is reasonable for a linear baseline.

- Accuracy alone is not sufficient, but provides context.

📌 This confirms the model is better than chance (≈33%) but far from perfect.


**2) Balanced Accuracy ≈ 45.8%**

Balanced accuracy is the average recall across all three classes.

**Interpretation:**

- The model identifies each class moderately better than random

- Performance is not dominated by the majority class (NO)

📌 This shows that using class_weight="balanced" is working as intended.


** 3) Macro F1 ≈ 0.435 **

**Macro F1:**

- Computes F1 separately per class

- Treats <30, >30, and NO equally

**Interpretation:**

- Performance on minority readmission classes is still weak

- Particularly sensitive to errors on <30

- This metric highlights the core difficulty of the task

📌 This is one of the most important metrics to report for this dataset.

** 4) Weighted F1 ≈ 0.515 **

**Weighted F1:**

- Weights each class by its frequency

- Reflects overall practical performance

**Interpretation:**

- Stronger performance on the majority class inflates this score

- The gap between weighted F1 and macro F1 clearly exposes class imbalance effects

📌 This contrast is good analysis material, not a weakness.


**5) Low standard deviation across folds**

All standard deviations are ≈ 0.004–0.005, which is small.

This means:

- The model behaves consistently

- Results are not sensitive to how the data is split

- Generalisation performance is stable

📌 Stability strengthens the credibility of your evaluation.

### **(2) Cross-Validation: V2**  -> With VarianceThreshold and StandardScaler in preprocessing

Removed low variance (nearly constant) features in preprocessing using VarianceThreshold.

-> StandardScaler is kept

-> Added VarianceThreshold(0.01) to remove features close to constants

Also:

 -> Hyper-Parameter C specified
 
 -> Comparing training and validation scores


 Questions?: 
 - Which order to apply StandardScaler and VarianceThreshold before training? 
 - Are StandardScaler and VarianceThreshold compatible for preprocessing?
 
 Answer:
 - StandardScaler should not be used before VarianceThreshold as it would try to make variance for all features as 1. 
 - RECOMMENDED VarianceThreshold -> StandardScaler

In [50]:
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score, balanced_accuracy_score

# Load cleaned data
DATA_PATH = Path("..") / "Dataset" / "clean_train_df.csv"
clean_df = pd.read_csv(DATA_PATH)

# Features X, and target labels y
X = clean_df.drop(columns=["readmitted"])
y = clean_df["readmitted"]   # values are: "NO", ">30", "<30"

# Build pipeline model
model = Pipeline(steps=[           # pipeline chains preprocessing + model into one object.
    ("low_var", VarianceThreshold(0.01)),   # step 2: removes features whose variance is <= 0.01                        
    ("scaler", StandardScaler()),           # step 1: standardises each feature to mean 0, std 1.                        
    ("clf", LogisticRegression(             # step 3: train logistic regression (the classifier) on the scaled data
            C = 0.1,                     # inverse of regularisation strength; smaller values specify stronger regularisation
            solver="lbfgs",              # optimisation method (default) to find best coefficients for multinomial + L2 regularisation
            max_iter=500,                # enough iterations for convergence in a high-dimensional dataset
            class_weight="balanced",     # increases the penalty for misclassifying minority classes (reweights classes inversely proportional to their frequency)
            n_jobs=None                  # lbfgs ignores n_jobs; OK to leave
            )
    )
])

# Cross-validation
cv = StratifiedKFold(
    n_splits=10, 
    shuffle=True, 
    random_state=42
)

# Define evaluation metrics - what to compute on each fold's validation set
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": make_scorer(balanced_accuracy_score),
    "f1_macro": make_scorer(f1_score, average="macro"),
    "f1_weighted": make_scorer(f1_score, average="weighted")
}

# Run cross-validation (= training + validation). There are 10 separate training runs.
cv_results_v2 = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)




In [51]:
# Summary results

# print("=== 10-Fold CV Logistic Regression Summary ===")
# for metric in scoring.keys():
#     scores = cv_results[f"test_{metric}"]
#     print(
#         f"{metric}: Mean={scores.mean():.4f}, Std={scores.std():.4f}"
    # )

print("=== V2: 10-Fold CV Logistic Regression Summary ===")
for metric in scoring.keys():
    train_scores = cv_results_v2[f"train_{metric}"]
    val_scores = cv_results_v2[f"test_{metric}"]

    print(
        f"\n{metric}: "
        f"Train mean ={train_scores.mean():.4f}, Train std ={train_scores.std():.4f} | "
        f"Val mean ={val_scores.mean():.4f}, Val std ={val_scores.std():.4f}"
    )

=== V2: 10-Fold CV Logistic Regression Summary ===

accuracy: Train mean =0.5022, Train std =0.0010 | Val mean =0.4987, Val std =0.0050

balanced_accuracy: Train mean =0.4636, Train std =0.0009 | Val mean =0.4579, Val std =0.0063

f1_macro: Train mean =0.4394, Train std =0.0010 | Val mean =0.4347, Val std =0.0047

f1_weighted: Train mean =0.5186, Train std =0.0008 | Val mean =0.5151, Val std =0.0051


**Results with "StandardScaler -> VarianceThreashold" order in the pipeline (Pipeline V1):**

Train:

| Metric              | Mean   | Std    |
|---------------------|--------|--------|
| Accuracy            | 0.5079 | 0.0009 |
| Balanced Accuracy   | 0.4779 | 0.0009 |
| F1 (Macro)          | 0.4505 | 0.0009 |
| F1 (Weighted)       | 0.5264 | 0.0008 |

Validation:

| Metric              | Mean   | Std    |
|---------------------|--------|--------|
| Accuracy            | 0.4951 | 0.0048 |
| Balanced Accuracy   | 0.4575 | 0.0054 |
| F1 (Macro)          | 0.4347 | 0.0043 |
| F1 (Weighted)       | 0.5145 | 0.0050 |

Overfitting analysis:

| Metric              | (Train - Val) Gap   | 
|---------------------|-------------------|
| Accuracy            | 0.0128            |


**RECOMMENDED Results with "VarianceThreashold -> StandardScaler" in the pipeline (Pipeline V2):**

Train:

| Metric              | Mean   | Std    |
|---------------------|--------|--------|
| Accuracy            | 0.5022 | 0.0010 |
| Balanced Accuracy   | 0.4636 | 0.0009 |
| F1 (Macro)          | 0.4394 | 0.0010 |
| F1 (Weighted)       | 0.5186 | 0.0008 |

Validation:

| Metric              | Mean   | Std    |
|---------------------|--------|--------|
| Accuracy            | 0.4987 | 0.0050 |
| Balanced Accuracy   | 0.4579 | 0.0063 |
| F1 (Macro)          | 0.4347 | 0.0047 |
| F1 (Weighted)       | 0.5151 | 0.0051 |


Overfitting analysis:

| Metric              | (Train - Val) Gap   | 
|---------------------|-------------------|
| Accuracy            | 0.0035            |




**-> Interpretation of Overfitting analysis (the train-vel gap)**

- Pipeline V1 has a larger train–validation gap

- Pipeline V2 has a *much smaller gap*

👉 **Pipeline V2 (VarianceThreashold -> StandardScaler) slightly generalises better**

- Removing low-variance features reduces overfitting
- Correct preprocessing improves generalisation, not raw training accuracy => exactly what we want to see

In [ ]:
# Scoring metric: [Results for 10 folds]
cv_results

### **(2) Cross-Validation: V3** -> with PCA and StandardScaler for preprocessing (no VarianceThreshold)
What PCA does:

- Finds directions with maximum variance

- Projects data into fewer dimensions

- Keeps most information, removes noise

Here 0.95 → keeps 95% of variance

**As a result introducing PCA**: reduces dimensionality and feature correlation but also increases bias, leading to slightly worse validation performance compared to targeted feature filtering. Increased model complexity does not necessarily lead to improved generalisation.

Question?:
- **Using both VarianceThreshold and PCA is unnecessary**. So remove VarianceThreashold when using PCA?

In [57]:
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score, balanced_accuracy_score

# Load cleaned data
DATA_PATH = Path("..") / "Dataset" / "clean_train_df.csv"
clean_df = pd.read_csv(DATA_PATH)

# Features X, and target labels y
X = clean_df.drop(columns=["readmitted"])
y = clean_df["readmitted"]   # values are: "NO", ">30", "<30"

# Build pipeline model
model = Pipeline(steps=[           # pipeline chains preprocessing + model into one object.
    # ("low_var", VarianceThreshold(0.01)),   # step 2: removes features whose variance is <= 0.01                        
    ("scaler", StandardScaler()),           # step 1: standardises each feature to mean 0, std 1.
    ("pca", PCA(n_components=0.95)),        # step 3: PCA to retain 95% variance
    ("clf", LogisticRegression(             # step 4: train logistic regression (the classifier) on the scaled data
            C = 0.1,                     # inverse of regularisation strength; smaller values specify stronger regularisation
            solver="lbfgs",              # optimisation method (default) to find best coefficients for multinomial + L2 regularisation
            max_iter=500,                # enough iterations for convergence in a high-dimensional dataset
            class_weight="balanced",     # increases the penalty for misclassifying minority classes (reweights classes inversely proportional to their frequency)
            n_jobs=None                  # lbfgs ignores n_jobs; OK to leave
            )
    )
])

# Cross-validation
cv = StratifiedKFold(
    n_splits=10, 
    shuffle=True, 
    random_state=42
)

# Define evaluation metrics - what to compute on each fold's validation set
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": make_scorer(balanced_accuracy_score),
    "f1_macro": make_scorer(f1_score, average="macro"),
    "f1_weighted": make_scorer(f1_score, average="weighted")
}

# Run cross-validation (= training + validation). There are 10 separate training runs.
cv_results_v3 = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)




In [59]:
# Summary results

# print("=== 10-Fold CV Logistic Regression Summary ===")
# for metric in scoring.keys():
#     scores = cv_results[f"test_{metric}"]
#     print(
#         f"{metric}: Mean={scores.mean():.4f}, Std={scores.std():.4f}"
    # )

print("=== V3: 10-Fold CV Logistic Regression with PCASummary ===")
for metric in scoring.keys():
    train_scores = cv_results_v3[f"train_{metric}"]
    val_scores = cv_results_v3[f"test_{metric}"]

    print(
        f"\n{metric}: "
        f"Train mean ={train_scores.mean():.4f}, Train std ={train_scores.std():.4f} | "
        f"Val mean ={val_scores.mean():.4f}, Val std ={val_scores.std():.4f}"
    )

=== V3: 10-Fold CV Logistic Regression with PCASummary ===

accuracy: Train mean =0.5016, Train std =0.0007 | Val mean =0.4902, Val std =0.0031

balanced_accuracy: Train mean =0.4727, Train std =0.0011 | Val mean =0.4552, Val std =0.0040

f1_macro: Train mean =0.4445, Train std =0.0009 | Val mean =0.4308, Val std =0.0027

f1_weighted: Train mean =0.5204, Train std =0.0007 | Val mean =0.5097, Val std =0.0031


**StandardScalar -> VarianceThreashold && PCA:**

Train:
| Metric            | Mean   | Std    |
| ----------------- | ------ | ------ |
| Accuracy          | 0.5016 | 0.0007 |
| Balanced Accuracy | 0.4727 | 0.0011 |
| F1 (Macro)        | 0.4445 | 0.0009 |
| F1 (Weighted)     | 0.5204 | 0.0007 |

Validation:

| Metric            | Mean   | Std    |
| ----------------- | ------ | ------ |
| Accuracy          | 0.4902 | 0.0031 |
| Balanced Accuracy | 0.4552 | 0.0040 |
| F1 (Macro)        | 0.4308 | 0.0027 |
| F1 (Weighted)     | 0.5097 | 0.0031 |

Overfitting analysis:

| Metric   | (Train − Val) Gap |
| -------- | ----------------- |
| Accuracy | 0.0114            |




**RECOMMENDED VarianceThreashold -> StandardScalar && PCA:**

Train:

| Metric            | Mean   | Std    |
| ----------------- | ------ | ------ |
| Accuracy          | 0.5016 | 0.0010 |
| Balanced Accuracy | 0.4624 | 0.0011 |
| F1 (Macro)        | 0.4383 | 0.0011 |
| F1 (Weighted)     | 0.5177 | 0.0009 |

Validation:

| Metric            | Mean   | Std    |
| ----------------- | ------ | ------ |
| Accuracy          | 0.4975 | 0.0049 |
| Balanced Accuracy | 0.4572 | 0.0055 |
| F1 (Macro)        | 0.4337 | 0.0041 |
| F1 (Weighted)     | 0.5136 | 0.0046 |


Overfitting analysis:

| Metric   | (Train − Val) Gap |
| -------- | ----------------- |
| Accuracy | 0.0041            |



**StandardScalar && PCA - only:**

Train:

| Metric            | Mean   | Std    |
| ----------------- | ------ | ------ |
| Accuracy          | 0.5016 | 0.0007 |
| Balanced Accuracy | 0.4727 | 0.0011 |
| F1 (Macro)        | 0.4445 | 0.0009 |
| F1 (Weighted)     | 0.5204 | 0.0007 |


Validation:
| Metric            | Mean   | Std    |
| ----------------- | ------ | ------ |
| Accuracy          | 0.4902 | 0.0031 |
| Balanced Accuracy | 0.4552 | 0.0040 |
| F1 (Macro)        | 0.4308 | 0.0027 |
| F1 (Weighted)     | 0.5097 | 0.0031 |

Overfitting analysis:
| Metric   | (Train − Val) Gap |
| -------- | ----------------- |
| Accuracy | 0.0114            |


### **(2) Cross-Validation: V4** -> Removed highly correlated features + VarianceThreshold and StandardScaler (no PCA)

**-> Logistic Regression assumes there should be no correlation or dependence between the input samples**

-> Hence, need to remove highly correlated features.

IMPORTANT ISSUE: data leakage risk (must fix)

- Correlations are computed using all data
- Validation folds influence feature selection

This is mild data leakage.

Correct approach (very important)

- Correlation filtering should happen inside the pipeline, per fold.
- But sklearn has no built-in correlation filter, so you have two options - keep outside & justify clearly or Solution below:
    - Solution: Wrap correlation filtering in a custom transformer, use inside the Pipeline ✅   

Question?: **Should PCA be added here?**
NO — and this is important

You are already:
- Removing highly correlated features
- Removing near-constant features
- Using regularisation

Adding PCA here would:
- Remove interpretability
- Duplicate correlation handling
- Increase bias

✔ **Correlation filtering + logistic regression is cleaner than PCA**

This is a strong methodological choice.

Question?: **Why this is much slower than your previous code?**

Before (fast):
- Correlation filtering done once
- Outside cross-validation

Now (slow but correct):
- Correlation filtering done inside the pipeline
- Recomputed for every fold

This is the price of avoiding data leakage.

So the slowdown is actually a sign that: ✅ Your code is now doing the right thing

In [64]:
from sklearn.base import BaseEstimator, TransformerMixin

class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.9):
        self.threshold = threshold

    def fit(self, X, y=None):
        corr = X.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        self.to_drop_ = [
            col for col in upper.columns
            if any(upper[col] > self.threshold)
        ]
        return self

    def transform(self, X):
        return X.drop(columns=self.to_drop_, errors="ignore")


In [65]:
import numpy as np

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score, balanced_accuracy_score


# Load cleaned data
DATA_PATH = Path("..") / "Dataset" / "clean_train_df.csv"
clean_df = pd.read_csv(DATA_PATH)

# Features X, and target labels y
X = clean_df.drop(columns=["readmitted"])
y = clean_df["readmitted"]   # values are: "NO", ">30", "<30"

"""
# Compute absolute correlation matrix (if corr(i, j) ≈ 1 => features carry almost the same information)
corr_matrix = X.corr().abs()

# Select upper triangle of correlation matrix (to avoid duplicate pairs and self-correlation, as correlation matrix is symmetric)
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Identify columns to drop (threshold = 0.9) (if a feature is highly correlated (> 0.9) with any other feature, drop it completely. This removes redundant dimensions.)
# Needed as dataset is high-dimensional and many medical features are correlated
to_drop = [
    column for column in upper.columns
    if any(upper[column] > 0.9)
]

print(f"Removing {len(to_drop)} correlated features")

X_uncorrelated = X.drop(columns=to_drop)
"""

# Build pipeline model (without PCA)
model = Pipeline(steps=[           # pipeline chains preprocessing + model into one object.
    ("corr_filter", CorrelationFilter(0.9)),                        
    ("low_var", VarianceThreshold(0.01)),  # step 2: removes features whose variance is ≤ 0.01             
    ("scaler", StandardScaler()),  # step 1: standardises each feature to mean 0, std 1.
    ("clf", LogisticRegression(    # step 3: train logistic regression (the classifier) on the scaled data
            # multi_class="multinomial",   # uses softmax + cross-entropy loss for 3 classes
            C = 0.1,                     # inverse of regularisation strength; smaller values specify stronger regularisation
            solver="lbfgs",              # optimisation method (default) to find best coefficients for multinomial + L2 regularisation
            max_iter=500,                # enough iterations for convergence in a high-dimensional dataset
            class_weight="balanced",     # increases the penalty for misclassifying minority classes (reweights classes inversely proportional to their frequency)
            n_jobs=None                  # lbfgs ignores n_jobs; OK to leave
            )
    )
])

# Cross-validation
cv = StratifiedKFold(
    n_splits=10, 
    shuffle=True, 
    random_state=42
)

# Define evaluation metrics - what to compute on each fold's validation set
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": make_scorer(balanced_accuracy_score),
    "f1_macro": make_scorer(f1_score, average="macro"),
    "f1_weighted": make_scorer(f1_score, average="weighted")
}

# Run cross-validation (= training + validation). There are 10 separate training runs.
cv_results = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)


In [66]:
# Summary results

# print("=== 10-Fold CV Logistic Regression Summary ===")
# for metric in scoring.keys():
#     scores = cv_results[f"test_{metric}"]
#     print(
#         f"{metric}: Mean={scores.mean():.4f}, Std={scores.std():.4f}"
    # )

print("=== V4: 10-Fold CV Logistic Regression with Uncorrelated Features Summary ===")
for metric in scoring.keys():
    train_scores = cv_results[f"train_{metric}"]
    val_scores = cv_results[f"test_{metric}"]

    print(
        f"\n{metric}: "
        f"Train mean ={train_scores.mean():.4f}, Train std ={train_scores.std():.4f} | "
        f"Val mean ={val_scores.mean():.4f}, Val std ={val_scores.std():.4f}"
    )

=== V4: 10-Fold CV Logistic Regression with Uncorrelated Features Summary ===

accuracy: Train mean =0.5023, Train std =0.0010 | Val mean =0.4981, Val std =0.0046

balanced_accuracy: Train mean =0.4633, Train std =0.0010 | Val mean =0.4570, Val std =0.0054

f1_macro: Train mean =0.4393, Train std =0.0010 | Val mean =0.4341, Val std =0.0042

f1_weighted: Train mean =0.5186, Train std =0.0009 | Val mean =0.5145, Val std =0.0047


**Result (took ~3 mins):**
accuracy: Train mean =0.5023, Train std =0.0010 | Val mean =0.4981, Val std =0.0046

balanced_accuracy: Train mean =0.4633, Train std =0.0010 | Val mean =0.4570, Val std =0.0054

f1_macro: Train mean =0.4393, Train std =0.0010 | Val mean =0.4341, Val std =0.0042

f1_weighted: Train mean =0.5186, Train std =0.0009 | Val mean =0.5145, Val std =0.0047

## Results from different Cross-Validation trainings:

#### 1) Baseline Logistic Regression (StandardScaler only)

- Training performance is consistently higher than validation performance across all metrics.

- A noticeable train–validation gap (≈ 1.3% in accuracy) indicates mild overfitting.

- Balanced accuracy and macro-F1 remain relatively low, reflecting class imbalance and the limited expressive power of a linear model.

#### 2) V2: Logistic Regression with Low-Variance Feature Removal

(VarianceThreshold → StandardScaler)

- When low-variance features are removed before scaling, the train–validation gap is reduced.

- Validation accuracy and weighted F1 slightly improve, while training performance decreases marginally.

- This behaviour indicates improved generalisation rather than memorisation.

- The results confirm that low-variance features were contributing noise rather than useful signal.

Key insight: preprocessing order matters — variance-based filtering is only meaningful in the original feature space.

#### 3) V3: Logistic Regression with PCA

(StandardScaler → PCA)

- Applying PCA leads to a reduction in both training and validation performance.

- While PCA reduces dimensionality and feature correlation, it also removes discriminative information relevant to class separation.

- The increased bias outweighs any variance reduction benefits for this dataset.

- PCA does not improve generalisation for logistic regression in this setting.

#### 4) V4: Logistic Regression with Correlation-Based Feature Removal (no PCA)

- Removing highly correlated features results in performance almost identical to the baseline.

- This suggests that multicollinearity was not a dominant limiting factor, likely due to the use of L2 regularisation.

- While coefficient stability and interpretability may improve, predictive performance remains unchanged.

#### Conclusions

- Logistic regression shows stable but limited performance across all configurations, indicating underfitting rather than overfitting.

- Properly ordered low-variance feature removal (VarianceThreshold → StandardScaler) improves generalisation by reducing noise without harming validation performance.

- PCA introduces additional bias and does not benefit logistic regression on this dataset.

- Correlation-based feature removal improves model robustness but does not significantly affect predictive accuracy.

- Overall, further gains are more likely to come from richer feature engineering or more expressive models rather than additional linear preprocessing.

